# 迭代：执行 → 观察 → 修复

这是入口笔记本。你将通过做智能体最普遍的那件事来学习 Managed Agents 的 API 面：尝试一下，读取发生了什么，再试一次。我们上传一个埋了两个缺陷的小包，告诉智能体让测试通过，然后看它运转这个循环：跑测试、读回溯、改代码、重跑，反复直到全绿。

在此过程中，你会看到 cookbook 其余各篇所依赖的全部 API 形态：智能体 / 环境 / 会话、文件挂载、事件流，以及 archive 调用。学完本笔记本，你就已经完成了端到端驱动一个智能体所需的一切。

## 概念

需要了解的三种资源：

- **智能体（Agent）**，一份可复用的配置（模型、系统提示词、工具）
- **环境（Environment）**，一个容器模板（依赖包、网络）
- **会话（Session）**，把一个智能体和一个环境绑定在一起，挂载智能体所需的文件，并产生一个事件流

智能体和环境各创建一次，在多个会话之间复用。每个会话是一次自包含的运行。

In [ ]:
import os
from pathlib import Path

from anthropic import Anthropic

MODEL = os.environ.get("COOKBOOK_MODEL", "claude-sonnet-4-6")

client = Anthropic()
FIXTURE = Path("example_data") / "iterate"

## 1. 创建智能体

系统提示词是有意写得简略的。我们希望智能体自己想出迭代循环，而不是照着一份逐步脚本执行；测试输出已经足够把任务说明白，不需要更多手把手指导。

`agent_toolset_20260401` 是内置工具集：bash、read、write、edit、glob、grep、web_fetch 和 web_search。把 `permission_policy` 设为 `always_allow`，智能体就能直接运行它们，不必往返请求确认。

In [ ]:
agent = client.beta.agents.create(
    name="cookbook-iterate",
    # Tells Anthropic which cookbook this agent came from. Safe to remove.
    metadata={"anthropic_cookbook": "claude-cookbooks/cma-iterate-fix-failing-tests"},
    model=MODEL,
    system=(
        "You are a debugging agent. Your job is to make failing tests pass. "
        "Run the tests, read the failures, fix the code, repeat until green. "
        "Stop when every assertion passes."
    ),
    tools=[
        {
            "type": "agent_toolset_20260401",
            "default_config": {
                "enabled": True,
                "permission_policy": {"type": "always_allow"},
            },
        }
    ],
)

## 2. 创建环境

环境是一个容器模板。`type: cloud` 在 Anthropic 的托管沙箱中运行。`networking: limited` 会阻止任意的出站流量；本笔记本完全不需要网络访问，所以我们把它锁死。

In [ ]:
env = client.beta.environments.create(
    name="cookbook-iterate-env",
    config={"type": "cloud", "networking": {"type": "limited"}},
)

## 3. 上传失败的测试

通过 Files API 上传文件以取回 ID。第 4 步中我们会把它们挂载到会话上。

`calc.py` 有两个埋入的缺陷，`test_calc.py` 有三个能抓到它们的断言。其中一个失败（`test_mean`）是另外两个的下游结果，这悄悄教会了智能体不要过度修复：`mean()` 内部调用 `add` 和 `divide`，所以那两处一旦修好，`test_mean` 就会自行通过，无需直接改动 `mean()`。

In [ ]:
calc_file = client.beta.files.upload(
    file=("calc.py", (FIXTURE / "calc.py").read_bytes(), "text/x-python")
)
test_file = client.beta.files.upload(
    file=("test_calc.py", (FIXTURE / "test_calc.py").read_bytes(), "text/x-python")
)
print(f"uploaded: {calc_file.id}, {test_file.id}")

## 4. 创建会话

会话把智能体和环境绑定，挂载智能体所需的文件，并启动一个全新的容器。`resources=` 就是在智能体启动前把数据放进容器的办法——编排笔记本演示了如何用同一个字段克隆一个 GitHub 仓库，而不是挂载单个文件。

文件挂载在 `/mnt/session/uploads/<mount_path>` 之下，该路径是只读的。智能体必须先把文件复制到可写目录（如 `/mnt/user` 或 `/tmp`）才能编辑它们；而你之后想取回的一切都放在 `/mnt/session/outputs/` 里。

In [ ]:
session = client.beta.sessions.create(
    environment_id=env.id,
    agent={"type": "agent", "id": agent.id, "version": agent.version},
    resources=[
        {"type": "file", "file_id": calc_file.id, "mount_path": "calc.py"},
        {"type": "file", "file_id": test_file.id, "mount_path": "test_calc.py"},
    ],
    title="Get the tests green",
)
print(f"session: {session.id}")

## 5. 驱动智能体并观察它的运作

两个步骤：发送一个带任务的 `user.message` 事件，然后读取事件流，直到智能体到达 `end_turn`。

这个流是一条服务器发送事件（SSE）连接。我们用它（而不是轮询，见文末的边栏），因为智能体会花约 30 秒做迭代，而我们想实时看到每一轮。

有两个需要内化的模式：

1. **先打开流，再发送。** `with` 块打开 SSE 连接；在块内 `send` 的任何内容都保证可被观察到。先发送后打开则有可能丢失在竞态窗口内触发的事件。
2. **在 `session.status_idle` 且 `stop_reason.type == "end_turn"` 时退出。** 会话任何时候在等待输入都会变为 idle，既包括回合结束，也包括自定义工具使用需要回应的时候。`stop_reason.type` 用于区分；`end_turn` 是我们的退出信号。门控笔记本演示了同一个循环中 `requires_action` 的那一侧。

In [ ]:
with client.beta.sessions.events.stream(session.id) as stream:
    client.beta.sessions.events.send(
        session_id=session.id,
        events=[
            {
                "type": "user.message",
                "content": [
                    {
                        "type": "text",
                        "text": (
                            "The tests in /mnt/session/uploads/test_calc.py are "
                            "failing. Copy both files into /mnt/user, iterate "
                            "on calc.py until every test passes, then write the "
                            "final calc.py to /mnt/session/outputs/calc.py. "
                            "pytest isn't installed here, run the assertions "
                            "directly with `python3 -c ...` instead."
                        ),
                    }
                ],
            }
        ],
    )
    print("--- iterate loop ---")
    for ev in stream:
        match ev.type:
            case "agent.message":
                for b in ev.content:
                    if b.type == "text":
                        print(b.text, end="")
            case "agent.tool_use":
                print(f"\n[{ev.name}]")
            case "session.status_idle" if ev.stop_reason and ev.stop_reason.type == "end_turn":
                break
            case "session.status_terminated":
                break

那个 `match ev.type:` 代码块就是规范的流式模式。本 cookbook 中其余每个笔记本都从 `utilities.py` 以 `stream_until_end_turn` 导入它，而不是重复那段循环。下面的 verify 步骤也用它。

`wait_for_idle_status` 是来自 `utilities.py` 的第二个辅助函数。它消化了第 7 步标注框里描述的那场竞态：即使流已经产出 `session.status_idle`，会话记录上服务端的 `status` 字段仍可能短暂显示 `running`，此时立刻调用 `archive()` 会返回 400。这个辅助函数只是轮询 `sessions.retrieve` 直到该字段稳定。先串流、紧接着归档的代码需要它。

In [ ]:
from utilities import stream_until_end_turn, wait_for_idle_status

## 6. 验证

不要轻信智能体自己的说法。独立地把每个断言再跑一遍，并打印最终的 `calc.py`。如果智能体在循环内最后一次运行与回合结束之间过度修复或改坏了什么，这一步能抓到。

In [ ]:
client.beta.sessions.events.send(
    session_id=session.id,
    events=[
        {
            "type": "user.message",
            "content": [
                {
                    "type": "text",
                    "text": (
                        "Re-run every assertion from "
                        "/mnt/session/uploads/test_calc.py one more time "
                        "against your final calc.py with `python3 -c ...` "
                        "to confirm they all pass, then cat the final "
                        "/mnt/session/outputs/calc.py."
                    ),
                }
            ],
        }
    ],
)
stream_until_end_turn(client, session.id)

## 7. 清理

归档（archive）是把会话、环境或智能体置为已完成的方式。它会拆掉任何活动容器，让该资源不再计入你的工作区配额，并把它从默认的 list 视图中隐藏，但会保留记录、配置和事件历史，用于审计以及供之后按 ID 取回该资源的人使用。如果你想彻底移除记录，多数资源还另外暴露了一个 `delete` 端点（operate 笔记本详细讲解了资源生命周期），但在一次运行的末尾，`archive` 几乎总是你想要的。

In [ ]:
wait_for_idle_status(client, session.id)
client.beta.sessions.archive(session.id)
client.beta.environments.archive(env.id)
client.beta.agents.archive(agent.id)
print("archived")

## 边栏：用轮询代替串流

当你想在智能体会花超过几秒的事情上看到实时进度时，第 5 步的串流模式是正确选择。对于更短的任务，或者对于不想持有长时 HTTP 连接的生产代码，你可以改用 `events.list` 轮询做同样的事：

```python
client.beta.sessions.events.send(session_id=..., events=[...])
while True:
    time.sleep(2)
    events = client.beta.sessions.events.list(session.id).data
    last = events[-1] if events else None
    if last and last.type == "session.status_terminated":
        break
    if (
        last
        and last.type == "session.status_idle"
        and last.stop_reason
        and last.stop_reason.type == "end_turn"
    ):
        break
# walk events to print agent.message text
```

取舍：

想看着智能体干活时，串流胜出。每一次工具使用、每一条部分消息、每一次状态转换，服务端一发出就立刻到达，而这正是开发新工作流时你想要的。代价是一条长时 SSE 连接：你的进程必须在整个回合期间保持存活，不能跨间隙暂停与恢复，而且关键时刻的一次网络抖动可能把你从流中途掉线，且没有干净的方式从断点恢复。

轮询在相反的情形下胜出。它无状态、能在进程重启后存活，并且能干净地配合不想保持连接打开的 webhook 处理程序与队列工作器。代价是时延和隐藏的进度：在你再次轮询之前什么都看不到，因此反馈受你的轮询间隔限制，而一个长回合在完成之前看起来就是一片沉默。

在智能体可能运行数分钟、而你的处理程序无法保持连接打开的生产部署中，你想要的就是轮询模式（或者它的生产版表亲，门控笔记本中展示的 `session.status_idled` webhook）。

## 下一步去哪里

迭代循环是智能体循环最简单的形态。本目录下配套的四篇笔记本建立在同样的 API 形态之上，演示了你能驱动的其他工作流：

- [﷐0﷑](CMA_orchestrate_issue_to_pr.ipynb) —— 在更长的工具链中保持状态的回合制智能体：读取一个 issue、写出修复、开一个 PR、从 CI 失败中恢复、回应一条评审意见，然后合并。
- [﷐0﷑](CMA_explore_unfamiliar_codebase.ipynb) —— 把一个智能体投放到它从未见过的仓库时的落地（grounding）模式，并埋了一个过期文档的陷阱。还演示了用 `sessions.resources.add` 往运行中的会话推送更多上下文。
- [﷐0﷑](CMA_gate_human_in_the_loop.ipynb) —— 面向人工介入工作流的自定义工具 `decide()` 与 `escalate()` 往返。覆盖 `requires_action` 的空闲弹回以及并行工具使用的去重。
- [﷐0﷑](CMA_operate_in_production.ipynb) —— 生产环境搭建的故事：由密钥库（vault）支撑的 MCP 凭据、用于在不保持长连接情况下实现 HITL 的 `session.status_idled` webhook，以及资源生命周期的 CRUD 动词。